# the GPT tokenizer, from scratch

This is the *Practice* step of `unit_09_tokenizer.md`. Do the Cold Attempt there first.

Work top to bottom. Each milestone is one cell of stubs followed by a grader cell.
The grader stops at your first failure so there is always exactly one thing in front of you.

**Rules of engagement**
- Don't open the lecture. Don't open the minbpe repo. No `tiktoken`.
- Stuck on an *idea* for 20 min → ask the coaching chat for a hint.
- Stuck on *Python syntax* → ask immediately, zero learning value in that.
- **Before you run a grader cell, say out loud what you expect to happen.**

Later milestones add methods to `BasicTokenizer` with `BasicTokenizer.name = name`. That's
just so you can build the class up one cell at a time instead of re-running one giant cell.

In [ ]:
import re
from test_tokenizer import grade

# --- given: training text --------------------------------------------------
with open("../data/tinyshakespeare.txt", encoding="utf-8") as f:
    shakespeare = f.read()
train_text = shakespeare[:4000]   # keep everything fast: seconds, not minutes


# --- given: a helper to look at a tokenization ------------------------------
def render(ids, vocab):
    """Print a token sequence with visible boundaries, e.g. [The][ tok][en]."""
    print("".join("[" + vocab[i].decode("utf-8", errors="replace") + "]" for i in ids))

## Milestone 1 — text → bytes → ints, and back

The model never sees characters. It sees a list of integers, and before any merging happens
those integers are the bytes of the text. Two functions: text to a list of ints, and the
reverse. The reverse has to survive input that is *not* valid text.

In [ ]:
def text_to_ids(text):
    """str -> list[int], one int per UTF-8 byte, each in 0..255.

    text_to_ids('hi') == [104, 105]
    """
    raise NotImplementedError


def ids_to_text(ids):
    """list[int] (each 0..255) -> str. Must not raise on byte sequences that are
    not valid UTF-8; such bytes become the replacement character U+FFFD."""
    raise NotImplementedError

In [ ]:
grade(text_to_ids, ids_to_text, upto=1)

## Milestone 2 — count consecutive pairs

The whole algorithm rests on one statistic: how often does each *adjacent* pair of ids occur?

In [ ]:
def get_stats(ids, counts=None):
    """Count consecutive pairs in ids.

    ids     list[int]
    counts  optional dict to accumulate into. If given, add to it and return the
            SAME dict; if None, start a fresh one.
    returns dict {(left, right): count}, keys inserted in order of first appearance.

    get_stats([1, 2, 3, 1, 2]) == {(1, 2): 2, (2, 3): 1, (3, 1): 1}
    """
    raise NotImplementedError

In [ ]:
grade(text_to_ids, ids_to_text, get_stats, upto=2)

## Milestone 3 — merge one pair everywhere

Given a pair and a fresh id, produce a new list where every occurrence of the pair is
replaced by that id. This is a five-line loop and the grader has opinions about every
edge of it. Write it, predict, then run.

In [ ]:
def merge(ids, pair, idx):
    """Return a NEW list: every consecutive occurrence of pair in ids replaced by idx.

    Scans left to right; once two elements are consumed they are not reused.
    merge([5, 6, 6, 7, 9, 1], (6, 7), 99) == [5, 6, 99, 9, 1]
    """
    raise NotImplementedError

In [ ]:
grade(text_to_ids, ids_to_text, get_stats, merge, upto=3)

## Milestone 4 — train: the BPE loop

Now the thing the lecture is about. The container is given. `train` is the idea.

**BasicTokenizer.train(text, vocab_size, verbose=False)**
- Holds, after it returns: `merges`, dict `{(left, right): new_id}`, insertion order ==
  training order, ids `256, 257, ...` one per merge; `vocab`, dict `{id: bytes}` with
  `vocab[i] == bytes([i])` for `i < 256` and, for a merged id, the bytes of the left
  part followed by the bytes of the right part. `len(vocab) == 256 + len(merges)`.
- Computes: `vocab_size - 256` times, the most common consecutive pair over the current
  ids (ties: the pair that appeared FIRST in the ids wins) replaced everywhere by the
  next unused id.
- Returns nothing; it fills the two fields. `train(text, 256)` leaves `merges == {}`.
  If nothing is left to count (fewer than two ids remain) it stops early with fewer
  merges than asked. `verbose=True` may print one line per merge.

The grader checks, in order: number of merges → ids consecutive from 256 → each merge is
the most common pair at that step (replayed with your own `get_stats` / `merge`) → vocab
has `256 + len(merges)` entries, raw bytes for the first 256 → merged entries spell
left + right → zero merges at vocab 256 → early stop on `"ab"` → 64 merges on 4000 chars
of Shakespeare compress it more than 1.5x.

In [ ]:
class BasicTokenizer:
    """Byte-level BPE, no pre-tokenization.

    Fields after train():
      merges  dict {(int, int): int}. Insertion order == training order. New ids
              start at 256 and go up by one per merge.
      vocab   dict {int: bytes}. vocab[i] == bytes([i]) for i < 256; for a merged
              id it is the bytes of the left part followed by the bytes of the
              right part.
    """

    def __init__(self):
        self.merges = {}
        self.vocab = {i: bytes([i]) for i in range(256)}
        self.special_tokens = {}   # used in milestone 7

    def train(self, text, vocab_size, verbose=False):
        """Learn (vocab_size - 256) merges from text.

        Each step: count pairs over the current ids, take the most common one
        (ties: the pair that appeared FIRST in the ids wins), replace it with the
        next unused id. Stop early if there is nothing left to count.
        verbose=True may print one line per merge.
        """
        raise NotImplementedError

In [ ]:
grade(text_to_ids, ids_to_text, get_stats, merge, BasicTokenizer, upto=4)

In [ ]:
# look at what it learned
tok = BasicTokenizer()
tok.train(train_text, 300, verbose=True)
for i in range(256, 300):
    print(i, tok.vocab[i])

## Milestone 5 — encode and decode

**BasicTokenizer.decode(ids)**
- Reads `self.vocab`. Every id in `ids` is a key of it.
- Computes: the concatenation of `vocab[i]` over `ids`, as one `str`.
- Returns `str`. `decode([104, 105]) == 'hi'`; `decode([]) == ''`. Byte sequences that
  are not valid UTF-8 must not raise: `decode([128]) == '\ufffd'`.

**BasicTokenizer.encode(text)**
- Reads `self.merges`.
- Computes: the id sequence training would have arrived at on this text — the bytes of
  `text` with the learned merges applied in the priority order they were learned, even
  when several different merges are possible at once.
- Returns `list[int]`, every id a key of `self.vocab`. `encode('') == []`,
  `encode('x') == [120]`, `decode(encode(s)) == s` for any `s`, and
  `encode(training_text)` equals the id list training ended with.

The grader checks, in order: `decode([104, 105])` → decode returns `str` → decode survives
invalid UTF-8 → `encode('')` and `encode('x')` → ids in vocab and round trip on four
strings → unseen text got shorter → exact ids on unseen text match the reference →
`encode(TRAIN_TEXT)` matches training's final ids.

In [ ]:
def decode(self, ids):
    """list[int] -> str. Every id is a key of self.vocab. Invalid UTF-8 must not raise."""
    raise NotImplementedError


def encode(self, text):
    """str -> list[int], using self.merges.

    Must reproduce exactly the sequence training would have arrived at on the same
    text: the merges have a priority order, and that order has to be respected
    even when several different merges are possible at once.
    """
    raise NotImplementedError


BasicTokenizer.decode = decode
BasicTokenizer.encode = encode

In [ ]:
grade(text_to_ids, ids_to_text, get_stats, merge, BasicTokenizer, upto=5)

In [ ]:
tok = BasicTokenizer()
tok.train(train_text, 320)
ids = tok.encode("Speak, speak. What is it you would speak of?")
render(ids, tok.vocab)
print(tok.decode(ids))

## Milestone 6 (stretch) — regex pre-tokenization

GPT-2 does not run BPE on the raw text. It first splits the text into chunks with a regex
so that, for example, a word and the punctuation after it can never become one token.
The pattern is given (simplified so it works with the standard `re` module). You wire
it in.

**RegexTokenizer(pattern=SPLIT_PATTERN)**, subclass of `BasicTokenizer`
- Holds: everything `BasicTokenizer` holds, plus `pattern`, the split regex as a `str`,
  the given one unchanged by default.
  `re.findall(tok.pattern, "Hello world's 123 done!!  ok") ==
  ['Hello', ' world', "'s", ' 123', ' done', '!!', ' ', ' ok']`.

**RegexTokenizer.train(text, vocab_size, verbose=False)**
- Same contract as `BasicTokenizer.train` (same fields, same ids, same tie rule, same
  early stop), except that the "current ids" are the chunks of
  `re.findall(self.pattern, text)`, each its own id list: pair counts are pooled over
  all chunks, a merge is applied inside every chunk, and a pair is never counted across
  a chunk boundary. Consequence: no `vocab` entry above 255 contains a space between two
  non-space characters.
- Returns nothing.

**RegexTokenizer.encode(text)**
- Computes: `re.findall(self.pattern, text)`, each chunk encoded on its own exactly as
  `BasicTokenizer.encode` would, concatenated in order.
- Returns `list[int]`, same edge cases as `BasicTokenizer.encode`. No output token spans
  a boundary training never saw: after training on runs of four spaces,
  `encode('  x') == [32, 32, 120]`.

`decode` is inherited unchanged.

The grader checks, in order: `.pattern` is a `str` and splits the sample as above →
merge order matches the reference (replayed per chunk) → vocab contents → no learned
token spans a space between two words → decode / encode edge cases and exact ids on
unseen text → `encode('  x')` after training on space runs → `encode('the cat')` never
glues across the space.

In [ ]:
# GPT-2's real pattern uses \p{L} and \p{N} (any Unicode letter / number), which need
# the third-party `regex` module. This is the same shape rewritten for `re`:
#   [^\W\d_]  ~  \p{L}        \d  ~  \p{N}        [^\s\w]  ~  [^\s\p{L}\p{N}]
SPLIT_PATTERN = r"""'s|'t|'re|'ve|'m|'ll|'d| ?[^\W\d_]+| ?\d+| ?[^\s\w]+|\s+(?!\S)|\s+"""

print(re.findall(SPLIT_PATTERN, "Hello world's 123 done!!  ok"))


class RegexTokenizer(BasicTokenizer):
    """BasicTokenizer, but text is split into chunks by self.pattern first.

    Merges never cross a chunk boundary, in training or in encoding.
    """

    def __init__(self, pattern=SPLIT_PATTERN):
        super().__init__()
        self.pattern = pattern

    def train(self, text, vocab_size, verbose=False):
        """Same contract as BasicTokenizer.train, but pair statistics are pooled over
        the chunks of re.findall(self.pattern, text) and merges are applied per chunk."""
        raise NotImplementedError

    def encode(self, text):
        """Split text into chunks, encode each chunk on its own, concatenate."""
        raise NotImplementedError

In [ ]:
grade(text_to_ids, ids_to_text, get_stats, merge, BasicTokenizer, RegexTokenizer, upto=6)

## Milestone 7 (stretch) — special tokens and compression ratio

**RegexTokenizer.encode_special(text)**
- Reads `self.special_tokens`, dict `{str: int}`, set by the caller, empty by default.
- Computes: `text` cut at every occurrence of a special-token string; each special
  occurrence becomes its id, each stretch of ordinary text between them becomes
  `self.encode(stretch)`; concatenated in text order.
- Returns `list[int]`. With no special tokens registered it is identical to
  `encode(text)`. With `special_tokens == {'<|endoftext|>': 300}`:
  `encode_special('<|endoftext|>') == [300]`, and adjacent specials with an empty
  stretch between them add nothing: `encode_special('<|endoftext|><|endoftext|>x') ==
  [300, 300, 120]`. Plain `encode` never recognises specials:
  `300 not in encode('<|endoftext|>')`.

**compression_ratio(text, ids)**, a plain function
- Computes `len(text.encode('utf-8')) / len(ids)`.
- Returns a `float`. `compression_ratio('hello', [1, 2, 3, 4, 5]) == 1.0`;
  `compression_ratio('😀', [1, 2]) == 2.0`, bytes not characters.

The grader checks, in order: `encode_special == encode` with no specials → a lone
special → specials mixed with text → adjacent specials → plain text → plain `encode`
ignores specials → three `compression_ratio` values → `BasicTokenizer` at vocab 320
compresses the Shakespeare slice more than 1.5x.

**The grade cell below passes `skip=(7,)`.** Milestone 6 is *not* in the skip tuple
because milestone 7 depends on it: `encode_special` calls `RegexTokenizer.encode`, and
the grader trains a `RegexTokenizer` and reads its `.pattern` before touching anything
from this milestone. Milestone 7 is the last one, so nothing depends on it; remove `7`
from the tuple when you want this milestone graded.

**Predict before you run the last cell of this milestone.** Same 4000 characters of
Shakespeare, same vocab size 320. Which compresses more, `BasicTokenizer` or
`RegexTokenizer`, and by roughly how much? Write your answer and reasoning in the Notes
section of the unit note *before* running it. The gap is the lesson.

In [ ]:
def encode_special(self, text):
    """Like encode, but each occurrence of a key of self.special_tokens becomes its
    id, and only the text between special tokens goes through ordinary encode.
    With no special tokens registered this is identical to encode."""
    raise NotImplementedError


RegexTokenizer.encode_special = encode_special


def compression_ratio(text, ids):
    """len(utf-8 bytes of text) / len(ids), as a float."""
    raise NotImplementedError

In [ ]:
grade(text_to_ids, ids_to_text, get_stats, merge, BasicTokenizer, RegexTokenizer, compression_ratio,
      skip=(7,))

In [ ]:
# prediction first, then run
unseen = shakespeare[4000:8000]
for cls in (BasicTokenizer, RegexTokenizer):
    t = cls()
    t.train(train_text, 320)
    print(f"{cls.__name__:15s} train {compression_ratio(train_text, t.encode(train_text)):.3f}x"
          f"   unseen {compression_ratio(unseen, t.encode(unseen)):.3f}x")

## Your own tokenizer, from memory

Close everything above (collapse the cells, don't scroll). In the cell below, write a
`train` / `encode` / `decode` triple from scratch as plain functions — no `get_stats`, no
`merge` reused from above, retype them. Then train on `train_text` with vocab 300, encode
a line you make up, decode it back, and check `tok.encode(line) == your_encode(line)`
against the graded version above. If they differ, find out why before looking anything up.

In [ ]:
def my_train(text, vocab_size):
    ...


def my_encode(text, merges):
    ...


def my_decode(ids, vocab):
    ...


## Scratch

Space to poke at things. `render(ids, tok.vocab)` shows where the boundaries fall.
Try: numbers, a word with and without a leading space, the same word capitalised, an emoji.